### **Data** I have got in Q3 i have uploaded to drive and fetching from there

In [2]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

drive_npz_path = "/content/drive/MyDrive/PA3/sd_dataset.npz"

os.makedirs("outputs", exist_ok=True)
local_npz_path = "outputs/sd_dataset.npz"

assert os.path.exists(drive_npz_path), f"File not found at {drive_npz_path}"
shutil.copy2(drive_npz_path, local_npz_path)
print("from drive to here:", local_npz_path)

Mounted at /content/drive
Copied successfully to: outputs/sd_dataset.npz


In [28]:

import os
import time
import json
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

DATA_FILE = "outputs/sd_dataset.npz"
OUT_DIR = "outputs_skip"

SEED = 42
HISTORY = 8            # 8 input steps = 2 hours
HORIZON = 4            # 4 output steps = 1 hour
HIDDEN = 32            # hidden feature size d
EMB_DIM = 10           # node embedding size d_e for the adaptive graph
ALPHA = 0.5            # for hybrid graph ALPHA * fixed + (1 - ALPHA) * adaptive
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
TRAIN_STRIDE = 1       # 1 = use every training window, 2 = every second window, ...
SKIP_CONNECTION = True    # False = exactly H = ReLU(A Z W_g). True = add Z to H.
MODELS_TO_RUN = ["temporal", "fixed", "adaptive", "hybrid"]
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
report_lines = []

def log(text=""):
    print(text, flush=True)
    report_lines.append(str(text))

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



### 4 (a) Preparing Forecasting Windows

In [29]:
# defining forecasting windows, splits and normalisation
data = np.load(DATA_FILE, allow_pickle=True)
flow = data["flow"].astype(np.float32)
observed = data["observed"].astype(bool)
adj_raw = data["adj"].astype(np.float32)
times = pd.to_datetime(data["timestamps"])
T, N = flow.shape

log("=" * 60)
log("Q4(a) Windows, splits, normalisation")
log("=" * 60)
log("Data shape (T, N): " + str(flow.shape))

train_rows = np.where(times.year == 2020)[0]
val_rows = np.where((times.year == 2021) & (times.month <= 3))[0]
test_rows = np.where((times.year == 2021) & (times.month >= 4))[0]


def make_anchors(rows):
    # as t is the last input step. The window uses t-7 ... t for input
    # and t+1 ... t+4 for the target. so all 12 steps must stay inside the split.
    first = rows[0]
    last = rows[-1]
    return np.arange(first + HISTORY - 1, last - HORIZON + 1)

train_anchors = make_anchors(train_rows)[::TRAIN_STRIDE]
val_anchors = make_anchors(val_rows)
test_anchors = make_anchors(test_rows)

log("Train period: " + str(times[train_rows[0]]) + " to " + str(times[train_rows[-1]]))
log("Val period:   " + str(times[val_rows[0]]) + " to " + str(times[val_rows[-1]]))
log("Test period:  " + str(times[test_rows[0]]) + " to " + str(times[test_rows[-1]]))
log("Windows -> train: " + str(len(train_anchors)) + ", val: " + str(len(val_anchors))
    + ", test: " + str(len(test_anchors)))

# normalisation statistics come from the training rows only (year 2020).
# each sensor has its own mean and std, because busy and quiet roads
# have very different flow levels.
train_values = flow[train_rows]
mean_np = train_values.mean(axis=0)              # shape (N,)
std_np = train_values.std(axis=0)                # shape (N,)
std_np[std_np < 1e-6] = 1.0                      # a constant sensor would divide by zero
log("Per-sensor training mean: min " + str(round(float(mean_np.min()), 2))
    + ", average " + str(round(float(mean_np.mean()), 2))
    + ", max " + str(round(float(mean_np.max()), 2)))
log("Per-sensor training std:  min " + str(round(float(std_np.min()), 2))
    + ", average " + str(round(float(std_np.mean()), 2))
    + ", max " + str(round(float(std_np.max()), 2)))

flow_norm = ((flow - mean_np) / std_np).astype(np.float32)

#same statistics as tensors with shape (1, 1, N), for use inside the model code.
MEAN = torch.from_numpy(mean_np.astype(np.float32)).reshape(1, 1, N).to(device)
STD = torch.from_numpy(std_np.astype(np.float32)).reshape(1, 1, N).to(device)

x_offsets = np.arange(-(HISTORY - 1), 1)        # -7 ... 0
y_offsets = np.arange(1, HORIZON + 1)           #  1 ... 4



Q4(a) Windows, splits, normalisation
Data shape (T, N): (70176, 716)
Train period: 2020-01-01 00:00:00 to 2020-12-31 23:45:00
Val period:   2021-01-01 00:00:00 to 2021-03-31 23:45:00
Test period:  2021-04-01 00:00:00 to 2021-12-31 23:45:00
Windows -> train: 35125, val: 8629, test: 26389
Per-sensor training mean: min 0.0, average 584.52, max 1360.66
Per-sensor training std:  min 1.0, average 382.46, max 848.43


### 4(b) Forecasting Architecture model

In [30]:

class STModel(nn.Module):
    def __init__(self, mode):
        super().__init__()
        self.mode = mode

        # temporal encoder (shared by all sensors)
        self.conv1 = nn.Conv1d(1, HIDDEN, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(HIDDEN, HIDDEN, kernel_size=3, padding=1)
        self.to_vector = nn.Linear(HIDDEN * HISTORY, HIDDEN)

        # graph convolution weights W_g
        if mode != "temporal":
            self.W_g = nn.Linear(HIDDEN, HIDDEN, bias=False)

        # learnable node embeddings for the adaptive graph
        if mode == "adaptive" or mode == "hybrid":
            self.E1 = nn.Parameter(torch.randn(N, EMB_DIM))
            self.E2 = nn.Parameter(torch.randn(N, EMB_DIM))

        # forecast head
        self.head = nn.Linear(HIDDEN, HORIZON)

    def adaptive_adjacency(self):
        # A_adaptive = Softmax(ReLU(E1 E2^T)), softmax over each row
        scores = torch.relu(self.E1 @ self.E2.T)
        return torch.softmax(scores, dim=1)

    def get_adjacency(self):
        if self.mode == "fixed":
            return A_fixed
        if self.mode == "adaptive":
            return self.adaptive_adjacency()
        if self.mode == "hybrid":
            # matrices have rows that sum to 1 before they are mixed.
            return ALPHA * A_row + (1.0 - ALPHA) * self.adaptive_adjacency()
        return None

    def forward(self, x):
        # x has shape (B, 8, N)
        B = x.shape[0]

        # temporal encoder: every sensor is one short time series
        h = x.permute(0, 2, 1).reshape(B * N, 1, HISTORY)
        h = torch.relu(self.conv1(h))
        h = torch.relu(self.conv2(h))
        h = h.reshape(B * N, HIDDEN * HISTORY)
        z = torch.relu(self.to_vector(h))
        z = z.reshape(B, N, HIDDEN)

        # spatial graph convolution: H = ReLU(A Z W_g)
        if self.mode == "temporal":
            hidden = z
        else:
            A = self.get_adjacency()
            hidden = torch.relu(torch.matmul(A, self.W_g(z)))
            if SKIP_CONNECTION:
                hidden = hidden + z

        # forecast head
        out = self.head(hidden)
        return out.permute(0, 2, 1)



In [31]:
def get_batch(anchors):
    # x: normalised inputs (B, 8, N)
    # y: targets in original units (B, 4, N)
    # m: True where the target is a real measurement (B, 4, N)
    x = flow_norm[anchors[:, None] + x_offsets]
    y = flow[anchors[:, None] + y_offsets]
    m = observed[anchors[:, None] + y_offsets]
    x = torch.from_numpy(x).to(device)
    y = torch.from_numpy(y).to(device)
    m = torch.from_numpy(m).to(device)
    return x, y, m

#### 5(a) Fixed adjacency normalisation


In [32]:
log()
log("Q5(a) Fixed adjacency normalisation")
log("-" * 60)

# A + I : I remove any existing diagonal first, so each self-loop is exactly 1.
A_tilde = adj_raw.copy()
np.fill_diagonal(A_tilde, 0.0)
A_tilde = A_tilde + np.eye(N, dtype=np.float32)
degree = A_tilde.sum(axis=1)

# Row normalisation D^-1 (A + I). Every row sums to 1.
A_row = A_tilde / degree[:, None]

is_symmetric = bool(np.allclose(adj_raw, adj_raw.T, atol=1e-6))
if is_symmetric:
    # Symmetric normalisation D^-1/2 (A + I) D^-1/2.
    d_inv_sqrt = 1.0 / np.sqrt(degree)
    A_fixed = d_inv_sqrt[:, None] * A_tilde * d_inv_sqrt[None, :]
    log("The road graph is symmetric. Using D^-1/2 (A+I) D^-1/2.")
else:
    # The graph is directed, so I use the row (random-walk) normalisation.
    A_fixed = A_row
    log("The road graph is NOT symmetric. Using row normalisation D^-1 (A+I).")

A_fixed = torch.from_numpy(A_fixed.astype(np.float32)).to(device)
A_row = torch.from_numpy(A_row.astype(np.float32)).to(device)




Q5(a) Fixed adjacency normalisation
------------------------------------------------------------
The road graph is NOT symmetric. Using row normalisation D^-1 (A+I).


### defining training and test functions

In [34]:

def masked_mae_loss(pred_norm, y, m):
    # Loss in normalised units. Filled (not real) targets are ignored.
    y_norm = (y - MEAN) / STD
    error = torch.abs(pred_norm - y_norm)
    return (error * m).sum() / m.sum().clamp(min=1)


def evaluate(model, anchors, keep_sensors=None):
    # calculate MAE and RMSE in the original traffic-flow units.
    model.eval()
    abs_sum = torch.zeros(HORIZON, device=device)
    sq_sum = torch.zeros(HORIZON, device=device)
    count = torch.zeros(HORIZON, device=device)
    kept_pred = []
    kept_true = []
    with torch.no_grad():
        for start in range(0, len(anchors), 256):
            x, y, m = get_batch(anchors[start:start + 256])
            pred = model(x) * STD + MEAN
            diff = (pred - y) * m
            abs_sum += diff.abs().sum(dim=(0, 2))
            sq_sum += (diff ** 2).sum(dim=(0, 2))
            count += m.sum(dim=(0, 2))
            if keep_sensors is not None:
                kept_pred.append(pred[:, :, keep_sensors].cpu().numpy())
                kept_true.append(y[:, :, keep_sensors].cpu().numpy())
    mae = float(abs_sum.sum() / count.sum())
    rmse = float(torch.sqrt(sq_sum.sum() / count.sum()))
    mae_per_step = (abs_sum / count).cpu().numpy()
    if keep_sensors is not None:
        return mae, rmse, mae_per_step, np.concatenate(kept_pred), np.concatenate(kept_true)
    return mae, rmse, mae_per_step


def train_one_model(mode):
    log()
    log("-" * 60)
    log("Training model: " + mode)
    log("-" * 60)
    set_seed(SEED)
    model = STModel(mode).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    log("Trainable parameters: " + str(n_params))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    rng = np.random.default_rng(SEED)
    best_val = float("inf")
    best_state = None
    history = []
    train_time = 0.0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        order = rng.permutation(train_anchors)
        loss_sum = 0.0
        n_batches = 0
        for start in range(0, len(order), BATCH_SIZE):
            x, y, m = get_batch(order[start:start + BATCH_SIZE])
            pred = model(x)
            loss = masked_mae_loss(pred, y, m)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
            n_batches += 1
        train_time += time.time() - t0
        val_mae, val_rmse, _ = evaluate(model, val_anchors)
        history.append({"epoch": epoch, "train_loss": loss_sum / n_batches,
                        "val_mae": val_mae, "val_rmse": val_rmse})
        log("epoch " + str(epoch) + "  train loss " + str(round(loss_sum / n_batches, 4))
            + "  val MAE " + str(round(val_mae, 3)) + "  val RMSE " + str(round(val_rmse, 3)))

        # keeping the weights with the best validation MAE.
        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}


    model.load_state_dict(best_state)

    val_mae, val_rmse, _ = evaluate(model, val_anchors)
    test_mae, test_rmse, test_steps, pred3, true3 = evaluate(model, test_anchors, PLOT_SENSORS)
    log("Best model -> val MAE " + str(round(val_mae, 3)) + ", val RMSE " + str(round(val_rmse, 3)))
    log("Best model -> TEST MAE " + str(round(test_mae, 3)) + ", TEST RMSE " + str(round(test_rmse, 3)))
    log("Test MAE per horizon step (15, 30, 45, 60 min): "
        + ", ".join(str(round(float(v), 3)) for v in test_steps))
    log("Training time: " + str(round(train_time, 1)) + " seconds")

    np.savez_compressed(os.path.join(OUT_DIR, "predictions_" + mode + ".npz"),
                        pred=pred3, true=true3, sensors=np.array(PLOT_SENSORS),
                        anchor_times=times[test_anchors].strftime("%Y-%m-%d %H:%M").values)
    if mode == "adaptive" or mode == "hybrid":
        learned = model.adaptive_adjacency().detach().cpu().numpy()
        np.save(os.path.join(OUT_DIR, "adj_learned_" + mode + ".npy"), learned)
    with open(os.path.join(OUT_DIR, "history_" + mode + ".json"), "w") as f:
        json.dump(history, f)

    adjacency_name = {"temporal": "None", "fixed": "Fixed",
                      "adaptive": "Adaptive", "hybrid": "Hybrid"}[mode]
    model_name = "Temporal-only" if mode == "temporal" else "STGNN"
    return {"Model": model_name, "Adjacency": adjacency_name,
            "Val MAE": round(val_mae, 3), "Val RMSE": round(val_rmse, 3),
            "Test MAE": round(test_mae, 3), "Test RMSE": round(test_rmse, 3),
            "Training time (s)": round(train_time, 1), "Parameters": n_params}


### Training

In [35]:
PLOT_SENSORS = [0, N // 2, N - 1]

log()
log("=" * 60)
log("Training configuration")
log("=" * 60)
log("Seed: " + str(SEED))
log("History: " + str(HISTORY) + ", horizon: " + str(HORIZON))
log("Hidden size: " + str(HIDDEN) + ", node embedding size: " + str(EMB_DIM))
log("Batch size: " + str(BATCH_SIZE) + ", epochs: " + str(EPOCHS)
    + ", learning rate: " + str(LEARNING_RATE))
log("Optimizer: Adam, loss: MAE on normalised flow")
log("Hybrid alpha: " + str(ALPHA) + ", train stride: " + str(TRAIN_STRIDE)
    + ", skip connection: " + str(SKIP_CONNECTION))
log("Normalisation: per-sensor z-score with 2020 statistics")
log("Device: " + str(device))
log("Machine: " + platform.platform())
log("Processor: " + str(platform.processor()) + ", CPU cores: " + str(os.cpu_count()))
log("PyTorch: " + torch.__version__ + ", NumPy: " + np.__version__)
if torch.cuda.is_available():
    log("GPU: " + torch.cuda.get_device_name(0))

results = []
for mode in MODELS_TO_RUN:
    results.append(train_one_model(mode))
    pd.DataFrame(results).to_csv(os.path.join(OUT_DIR, "q4_q5_results.csv"), index=False)

log()
log("=" * 60)
log("Final comparison (test period April-December 2021)")
log("=" * 60)
log(pd.DataFrame(results).to_string(index=False))

with open(os.path.join(OUT_DIR, "q4_q5_log.txt"), "w") as f:
    f.write("\n".join(report_lines))
print("Saved q4_q5_results.csv and q4_q5_log.txt")


Training configuration
Seed: 42
History: 8, horizon: 4
Hidden size: 32, node embedding size: 10
Batch size: 64, epochs: 10, learning rate: 0.001
Optimizer: Adam, loss: MAE on normalised flow
Hybrid alpha: 0.5, train stride: 1, skip connection: True
Normalisation: per-sensor z-score with 2020 statistics
Device: cuda
Machine: Linux-6.6.159+-x86_64-with-glibc2.39
Processor: x86_64, CPU cores: 2
PyTorch: 2.14.1+cu130, NumPy: 2.4.6
GPU: Tesla T4

------------------------------------------------------------
Training model: temporal
------------------------------------------------------------
Trainable parameters: 11588
epoch 1  train loss 0.1903  val MAE 57.083  val RMSE 91.526
epoch 2  train loss 0.1476  val MAE 56.079  val RMSE 90.111
epoch 3  train loss 0.1457  val MAE 55.602  val RMSE 89.856
epoch 4  train loss 0.1449  val MAE 55.335  val RMSE 89.184
epoch 5  train loss 0.1438  val MAE 54.849  val RMSE 88.452
epoch 6  train loss 0.1433  val MAE 56.096  val RMSE 90.555
epoch 7  train los

##  Question 6


In [40]:
# comparison table, prediction plots, learned-graph analysis.

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = "outputs_skip"
MODES = ["temporal", "fixed", "adaptive", "hybrid"]
LABELS = {"temporal": "Temporal-only", "fixed": "STGNN fixed",
          "adaptive": "STGNN adaptive", "hybrid": "STGNN hybrid"}

HORIZON_STEP = 4        # which forecast step to plot: 4 = 60 minutes ahead
PLOT_START = "2021-04-05 00:00"     # a Monday inside the test period
PLOT_DAYS = 3
TOP_K = 10              # strongest learned neighbours per sensor

report_lines = []


def log(text=""):
    print(text)
    report_lines.append(str(text))


# Q6(a) - Forecasting comparison table
log("=" * 60)
log("Q6(a) Forecasting comparison (test period April-December 2021)")
log("=" * 60)

# keep_default_na=False stops pandas from reading the word "None" as a missing value.
results = pd.read_csv(os.path.join(OUT_DIR, "q4_q5_results.csv"), keep_default_na=False)
log("| Model | Adjacency | MAE | RMSE | Training time (s) |")
log("|---|---|---|---|---|")
for _, r in results.iterrows():
    log("| " + str(r["Model"]) + " | " + str(r["Adjacency"]) + " | " + str(r["Test MAE"])
        + " | " + str(r["Test RMSE"]) + " | " + str(r["Training time (s)"]) + " |")
log()
log("Validation metrics (January-March 2021):")
log(results[["Model", "Adjacency", "Val MAE", "Val RMSE"]].to_string(index=False))

best_row = results.loc[results["Test MAE"].idxmin()]
base_mae = None
baseline_rows = results[results["Adjacency"] == "None"]
if len(baseline_rows) > 0:
    base_mae = float(baseline_rows["Test MAE"].iloc[0])
log()
log("Lowest test MAE: " + str(best_row["Model"]) + " / " + str(best_row["Adjacency"]))
if base_mae is not None:
    for _, r in results.iterrows():
        change = 100 * (float(r["Test MAE"]) - base_mae) / base_mae
        log("  " + str(r["Model"]) + " / " + str(r["Adjacency"]) + ": test MAE change vs temporal-only = "
            + str(round(change, 2)) + "%")

fig, ax = plt.subplots(figsize=(8, 4.5))
for mode in MODES:
    path = os.path.join(OUT_DIR, "history_" + mode + ".json")
    if not os.path.exists(path):
        continue
    with open(path) as f:
        history = json.load(f)
    epochs = [h["epoch"] for h in history]
    val = [h["val_mae"] for h in history]
    ax.plot(epochs, val, marker="o", linewidth=2, label=LABELS[mode])
ax.set_title("Validation MAE during training")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MAE (flow units)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "q6a_validation_curves.png"), dpi=150)
plt.close(fig)
log("Saved q6a_validation_curves.png")

#  predicted versus actual, one figure per model
meta = pd.read_csv(os.path.join(OUT_DIR, "sd_meta.csv"))
step = HORIZON_STEP - 1
for mode in MODES:
    path = os.path.join(OUT_DIR, "predictions_" + mode + ".npz")
    if not os.path.exists(path):
        continue
    p = np.load(path, allow_pickle=True)
    pred = p["pred"]                 # (windows, 4, 3)
    true = p["true"]
    sensors = p["sensors"]
    # The forecast made at anchor time t for step h belongs to time t + h * 15 minutes.
    target_times = pd.to_datetime(p["anchor_times"]) + pd.Timedelta(minutes=15 * HORIZON_STEP)
    start = pd.Timestamp(PLOT_START)
    end = start + pd.Timedelta(days=PLOT_DAYS)
    keep = (target_times >= start) & (target_times < end)

    fig, axes = plt.subplots(len(sensors), 1, figsize=(11, 7.5), sharex=True)
    for row_number, ax in enumerate(axes):
        s = int(sensors[row_number])
        ax.plot(target_times[keep], true[keep, step, row_number],
                color="black", linewidth=1.5, label="Actual")
        ax.plot(target_times[keep], pred[keep, step, row_number],
                color="tab:blue", linewidth=1.5, label="Predicted")
        sensor_mae = np.mean(np.abs(pred[keep, step, row_number] - true[keep, step, row_number]))
        ax.set_title("Sensor " + str(meta.iloc[s]["ID"]) + "  (" + str(meta.iloc[s]["Fwy"]) + " "
                     + str(meta.iloc[s]["Direction"]) + "),  MAE in this window = "
                     + str(round(float(sensor_mae), 1)), fontsize=10, loc="left")
        ax.set_ylabel("Flow")
        ax.grid(alpha=0.3)
    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Time")
    fig.suptitle(LABELS[mode] + ": predicted vs actual flow, " + str(15 * HORIZON_STEP)
                 + " minutes ahead")
    fig.tight_layout()
    name = "q6a_pred_vs_actual_" + mode + ".png"
    fig.savefig(os.path.join(OUT_DIR, name), dpi=150)
    plt.close(fig)
    log("Saved " + name)

# Q6(b) - Graph structure analysis
log()
log("=" * 60)
log("Q6(b) Learned graph versus road graph")
log("=" * 60)

data = np.load(os.path.join("outputs", "sd_dataset.npz"), allow_pickle=True)
road = data["adj"].astype(np.float32)
N = road.shape[0]
sensor_ids = [str(s) for s in data["sensor_ids"]]

road_edge = road != 0
np.fill_diagonal(road_edge, False)
off_diagonal = ~np.eye(N, dtype=bool)
road_density = road_edge.sum() / off_diagonal.sum()
log("Road graph: " + str(int(road_edge.sum())) + " edges without self-loops, density "
    + str(round(float(road_density), 4)))


def haversine_km(lat, lng):
    # Straight-line distance in km between every pair of sensors.
    lat = np.radians(lat)[:, None]
    lng = np.radians(lng)[:, None]
    dlat = lat - lat.T
    dlng = lng - lng.T
    a = np.sin(dlat / 2) ** 2 + np.cos(lat) * np.cos(lat.T) * np.sin(dlng / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


dist = haversine_km(meta["Lat"].values.astype(float), meta["Lng"].values.astype(float))
log("Mean distance between all sensor pairs: " + str(round(float(dist[off_diagonal].mean()), 2)) + " km")
log("Mean distance of road-graph edges: " + str(round(float(dist[road_edge].mean()), 2)) + " km")
log("Median distance of road-graph edges: " + str(round(float(np.median(dist[road_edge])), 2)) + " km")


def analyse_learned_graph(mode):
    path = os.path.join(OUT_DIR, "adj_learned_" + mode + ".npy")
    if not os.path.exists(path):
        return None
    L = np.load(path)
    log()
    log("--- Learned adaptive matrix from the " + mode + " model ---")

    # How much weight does a sensor give to itself?
    self_weight = np.diag(L)
    log("Mean self weight (diagonal): " + str(round(float(self_weight.mean()), 4))
        + "   (a uniform row would give " + str(round(1.0 / N, 4)) + ")")

    # How many neighbours does a row effectively use?
    entropy = -(L * np.log(L + 1e-12)).sum(axis=1)
    log("Effective neighbours per row (exp of entropy): mean "
        + str(round(float(np.exp(entropy).mean()), 1)) + " of " + str(N))
    log("Largest weight in a row: mean " + str(round(float(L.max(axis=1).mean()), 4)))

    # Top-K strongest learned neighbours of each sensor, self excluded.
    L_no_self = L.copy()
    np.fill_diagonal(L_no_self, -1.0)
    top = np.argsort(-L_no_self, axis=1)[:, :TOP_K]          # (N, K)
    rows = np.repeat(np.arange(N), TOP_K)
    cols = top.reshape(-1)

    # 1. Are the strongest learned connections also road-graph edges?
    in_road = road_edge[rows, cols]
    log("1. Top-" + str(TOP_K) + " learned edges that are also road-graph edges: "
        + str(round(100 * float(in_road.mean()), 2)) + "%")
    log("   Chance level (road-graph density): " + str(round(100 * float(road_density), 2)) + "%")

    # 2. Are learned connections between nearby sensors?
    d_top = dist[rows, cols]
    log("2. Mean distance of top-" + str(TOP_K) + " learned edges: "
        + str(round(float(d_top.mean()), 2)) + " km")
    log("   Median distance of top-" + str(TOP_K) + " learned edges: "
        + str(round(float(np.median(d_top)), 2)) + " km")
    log("   Share of top learned edges within 5 km: "
        + str(round(100 * float((d_top < 5).mean()), 2)) + "%")
    log("   Share of all sensor pairs within 5 km: "
        + str(round(100 * float((dist[off_diagonal] < 5).mean()), 2)) + "%")

    # 3. Learned connections that the road graph does not have.
    absent = ~in_road
    log("3. Top learned edges absent from the road graph: " + str(int(absent.sum()))
        + " of " + str(len(rows)) + " (" + str(round(100 * float(absent.mean()), 2)) + "%)")
    if absent.sum() > 0:
        log("   Mean distance of these absent edges: "
            + str(round(float(d_top[absent].mean()), 2)) + " km")
        weights = L[rows, cols]
        order = np.argsort(-np.where(absent, weights, -1.0))[:10]
        log("   Ten strongest learned edges that are not in the road graph:")
        for k in order:
            i = rows[k]
            j = cols[k]
            log("     " + sensor_ids[i] + " (" + str(meta.iloc[i]["Fwy"]) + " " + str(meta.iloc[i]["Direction"])
                + ") -> " + sensor_ids[j] + " (" + str(meta.iloc[j]["Fwy"]) + " " + str(meta.iloc[j]["Direction"])
                + "), weight " + str(round(float(weights[k]), 4))
                + ", distance " + str(round(float(dist[i, j]), 1)) + " km")
    return L


learned = {}
for mode in ["adaptive", "hybrid"]:
    L = analyse_learned_graph(mode)
    if L is not None:
        learned[mode] = L

#  adjacency visualisation: road graph next to the learned graphs
road_tilde = road.copy()
np.fill_diagonal(road_tilde, 0.0)
road_tilde = road_tilde + np.eye(N, dtype=np.float32)
road_row = road_tilde / road_tilde.sum(axis=1, keepdims=True)

panels = [("Road graph (row-normalised)", road_row)]
for mode in learned:
    panels.append(("Learned graph (" + mode + " model)", learned[mode]))

fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 5.5))
if len(panels) == 1:
    axes = [axes]
for ax, (title, M) in zip(axes, panels):
    top_value = float(np.percentile(M, 99.5))        # clip so that weak entries stay visible
    if top_value <= 0:
        top_value = float(M.max())
    im = ax.imshow(M, cmap="Blues", vmin=0.0, vmax=top_value, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Sensor index j")
    ax.set_ylabel("Sensor index i")
    fig.colorbar(im, ax=ax, label="Weight (clipped at 99.5th percentile)", shrink=0.85)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "q6b_learned_adjacency.png"), dpi=150)
plt.close(fig)
log()
log("Saved q6b_learned_adjacency.png")

with open(os.path.join(OUT_DIR, "q6_results.txt"), "w") as f:
    f.write("\n".join(report_lines))
print("Saved q6_results.txt")

Q6(a) Forecasting comparison (test period April-December 2021)
| Model | Adjacency | MAE | RMSE | Training time (s) |
|---|---|---|---|---|
| Temporal-only | None | 62.713 | 99.009 | 194.8 |
| STGNN | Fixed | 59.535 | 93.658 | 209.7 |
| STGNN | Adaptive | 54.854 | 86.485 | 213.7 |
| STGNN | Hybrid | 56.436 | 88.974 | 216.1 |

Validation metrics (January-March 2021):
        Model Adjacency  Val MAE  Val RMSE
Temporal-only      None   54.274    87.858
        STGNN     Fixed   51.448    82.714
        STGNN  Adaptive   45.814    74.199
        STGNN    Hybrid   48.328    78.178

Lowest test MAE: STGNN / Adaptive
  Temporal-only / None: test MAE change vs temporal-only = 0.0%
  STGNN / Fixed: test MAE change vs temporal-only = -5.07%
  STGNN / Adaptive: test MAE change vs temporal-only = -12.53%
  STGNN / Hybrid: test MAE change vs temporal-only = -10.01%
Saved q6a_validation_curves.png
Saved q6a_pred_vs_actual_temporal.png
Saved q6a_pred_vs_actual_fixed.png
Saved q6a_pred_vs_actual_adap